In [2]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
    full_finetuning=False
)

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX A4000. Num GPUs = 1. Max memory: 14.615 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu129. CUDA: 8.6. CUDA Toolkit: 12.9. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [4]:
import re
from datasets import load_dataset
from unsloth import FastLanguageModel
import torch

# 1. Load GSM8K Subset (Test split)
dataset = load_dataset("gsm8k", "main", split="test")
subset = dataset.select(range(50)) # Test with 50 samples first

def extract_answer(text):
    # GSM8K answers usually end with #### <number>
    # We look for the last number in the string as a fallback
    matches = re.findall(r"[-+]?\d*\.\d+|\d+", text.split("####")[-1] if "####" in text else text)
    return matches[-1] if matches else None

def run_benchmark(model, tokenizer, data_subset):
    FastLanguageModel.for_inference(model) # Enable 2x faster inference
    correct = 0
    results = []

    for i, item in enumerate(data_subset):
        question = item['question']
        gold_answer = extract_answer(item['answer'])
        
        # Consistent Prompting
        prompt = f"Question: {question}\nAnswer the question step-by-step. End your answer with 'The final answer is: #### <number>'."
        
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=256)
        prediction_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        predicted_answer = extract_answer(prediction_text)
        
        is_correct = (predicted_answer == gold_answer)
        if is_correct: correct += 1
        
        results.append({
            "question": question,
            "gold": gold_answer,
            "pred": predicted_answer,
            "correct": is_correct
        })
        print(f"[{i+1}/{len(data_subset)}] Correct: {is_correct} | Pred: {predicted_answer} | Gold: {gold_answer}")

    accuracy = (correct / len(data_subset)) * 100
    return accuracy, results

# 2. Benchmark the Models
# Run this once for base_model, then once for your fine_tuned_model
accuracy, detailed_results = run_benchmark(model, tokenizer, subset)
print(f"Final Accuracy: {accuracy}%")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

...

[47/50] Correct: False | Pred: 6 | Gold: 163


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[48/50] Correct: False | Pred: 5 | Gold: 800


Both `max_new_tokens` (=256) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[49/50] Correct: True | Pred: 8 | Gold: 8
[50/50] Correct: True | Pred: 30 | Gold: 30
Final Accuracy: 57.99999999999999%


In [5]:
del model
del tokenizer
torch.cuda.empty_cache()

In [8]:
model, tokenizer

(Qwen3ForCausalLM(
   (model): Qwen3Model(
     (embed_tokens): Embedding(151936, 2560, padding_idx=151669)
     (layers): ModuleList(
       (0): Qwen3DecoderLayer(
         (self_attn): Qwen3Attention(
           (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
           (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
           (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
           (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
           (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
           (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
           (rotary_emb): LlamaRotaryEmbedding()
         )
         (mlp): Qwen3MLP(
           (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
           (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
           (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
           (act_fn): SiLUActivation()
         

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)

Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [10]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template='qwen3-instruct'
)

In [11]:
from datasets import load_dataset

dataset = load_dataset('mlabonne/FineTome-100k', split="train")
dataset[0]

{'conversations': [{'from': 'human',
   'value': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'from': 'gpt

In [12]:
from unsloth.chat_templates import standardize_data_formats

dataset = standardize_data_formats(dataset)
dataset[0]

{'conversations': [{'role': 'user',
   'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'role': 'as

In [13]:
def format_prompts(examples):
    convos = examples['conversations']
    texts = [tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False) for convo in convos
    ]
    return {'text': texts}

dataset = dataset.map(format_prompts, batched=True)
dataset[0]

{'conversations': [{'role': 'user',
   'content': 'Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.'},
  {'role': 'as

In [14]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args = SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='linear',
        seed=3407,
        report_to="none"
    )
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [15]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n'
)

Map (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Unsloth: Removed 94 out of 100000 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [16]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB")
print(f"{start_gpu_memory} GB of memory reserved")

GPU = NVIDIA RTX A4000. Max memory = 14.615 GB
6.852 GB of memory reserved


In [29]:
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [31]:
import wandb
import os

# 1. Force-close any existing 'stale' sessions
try:
    wandb.finish()
except:
    pass

# 2. Kill zombie wandb processes that might be holding the 'mailbox' lock
os.system("pkill -9 -f wandb")

# 3. Test your internet connection to the W&B API from inside the container
print("Checking connectivity to W&B...")
os.system("curl -Is https://api.wandb.ai | head -n 1")

# 4. Set a longer timeout (default is often too short for remote cloud handshakes)
os.environ["WANDB_INIT_TIMEOUT"] = "300"

Checking connectivity to W&B...
HTTP/2 404 


In [35]:
import wandb

wandb.init(
    project="sample-fine-tuning",
    name="unsloth/Qwen3-4B-Instruct-2507", 
    settings=wandb.Settings(start_method="fork"),
    reinit=True,
    mode="offline",
    config={
        "learning_rate": 2e-4,
        "model": "unsloth/Qwen3-4B-Instruct-2507",
        "dataset": "mlabonne/FineTome-100k",
        "architecture": "QLoRA",
    }
)


MailboxClosedError: 

In [17]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 99,906 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.461075
2,1.434296
3,1.511298
4,1.174444
5,1.082929
6,0.799613
7,0.810148
8,0.601996
9,0.514432
10,0.885318


In [21]:
import re
from unsloth import FastLanguageModel

def extract_answer(text):
    # Extracts the final number, looking specifically for the GSM8K #### delimiter
    matches = re.findall(r"[-+]?\d*\.\d+|\d+", text.split("####")[-1] if "####" in text else text)
    return matches[-1] if matches else None

def run_benchmark(model, tokenizer, data_subset):
    FastLanguageModel.for_inference(model) # Enable 2x faster inference
    correct = 0
    results = []

    for i, item in enumerate(data_subset):
        question = item['question']
        gold_answer = extract_answer(item['answer'])
        
        # 1. Structure the input as a conversation dictionary
        messages = [
            {
                "role": "user", 
                "content": f"{question}\nAnswer the question step-by-step. End your answer with 'The final answer is: #### <number>'."
            }
        ]
        
        # 2. Apply the template (Injects <|im_start|> tags and the assistant trigger)
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True 
        )
        
        # 3. Tokenize and execute inference
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=2048)
        
        # 4. Decode and evaluate
        prediction_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        predicted_answer = extract_answer(prediction_text)
        
        is_correct = (predicted_answer == gold_answer)
        if is_correct: 
            correct += 1
        
        results.append({
            "question": question,
            "gold": gold_answer,
            "pred": predicted_answer,
            "correct": is_correct
        })
        
        print(f"[{i+1}/{len(data_subset)}] Correct: {is_correct} | Pred: {predicted_answer} | Gold: {gold_answer}")

    accuracy = (correct / len(data_subset)) * 100
    return accuracy, results

# Execution example:
accuracy, detailed_results = run_benchmark(model, tokenizer, subset)
print(f"Final Fine-Tuned Accuracy: {accuracy}%")

Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/50] Correct: False | Pred: 25 | Gold: 18


Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/50] Correct: True | Pred: 3 | Gold: 3


...

[47/50] Correct: True | Pred: 163 | Gold: 163


Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[48/50] Correct: True | Pred: 800 | Gold: 800


Both `max_new_tokens` (=2048) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[49/50] Correct: True | Pred: 8 | Gold: 8
[50/50] Correct: True | Pred: 30 | Gold: 30
Final Fine-Tuned Accuracy: 76.0%


In [29]:
messages = [
    {
        "role": "user",
        "content": """
        It takes 1 hour to dry 3 shirts outside in the sun on a hot sunny day.
        How long will it take to dry 30 shirts on the same hot sunny day outside in the sun?

        VERY IMPORTANT INSTRUCTION!!
        -----------------------------
        Assume I have a backyard of infinite size
        """
    }
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

In [30]:
from transformers import TextStreamer

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=4000,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True)
)

Both `max_new_tokens` (=4000) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 

1: Understand the problem
The problem states that it takes 1 hour to dry 3 shirts outside in the sun on a hot sunny day. We need to find out how long it will take to dry 30 shirts on the same hot sunny day outside in the sun, assuming we have a backyard of infinite size.

Step 2: Identify the relevant information
We know that it takes 1 hour to dry 3 shirts. We need to find out how long it will take to dry 30 shirts.

Step 3: Set up the equation
Since the number of shirts is directly proportional to the time it takes to dry them, we can set up the following equation:
Time to dry 3 shirts = 1 hour
Time to dry 30 shirts = x hours

Step 4: Solve the equation
To solve for x, we can use the following proportion:
3 shirts / 1 hour = 30 shirts / x hours
Cross-multiplying, we get:
3x = 30
Dividing both sides by 3, we get:
x = 10

Step 5: Answer the question
It will take 10 hours to dry 30 shirts on the same hot sunny day outside in the sun.<|im_end|>
